# Module 7 - Activity 2: Bank Loan Explainability (AI Explainability 360, rebuilt locally)

**Brief:** work through the IBM AIX360 bank-loan demo; see what explanation each *consumer type*
needs (data scientist, loan officer, bank customer); explore the **CEM** and **Protodash** explainers.
Then answer: *if the bank were legally obliged to explain every rejection, could you?*

**Why this notebook:** `aix360.mybluemix.net` is dead and the HELOC dataset needs registration.
We use **German Credit** (`credit-g`, 1,000 real loan applications) and implement the two explainers
from scratch - which is the point, because it forces you to see what they actually *do*.

### The one idea of AIX360
> **One decision. Four audiences. Four different explanations. No single explanation serves all.**

| Audience | The question they are actually asking | Explainer | Scope |
|---|---|---|---|
| **Loan officer** | "what drives approvals in general?" | interpretable model (rules/coefficients) | **global** |
| **Rejected customer** | "what would I have to change?" | **CEM** (pertinent negative) | **local** |
| **Data scientist** | "which features drove *this* score?" | **SHAP** | **local + global** |
| **Regulator / adjudicator** | "is this decision consistent with precedent?" | **Protodash** (prototypes) | **local** |


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

SEED = 42
np.random.seed(SEED)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
DATA = Path('dataset'); DATA.mkdir(exist_ok=True)

cache = DATA / 'german_credit.csv'
if cache.exists():
    raw = pd.read_csv(cache)
else:
    raw = fetch_openml('credit-g', version=1, as_frame=True).frame
    raw.to_csv(cache, index=False)

# target: 'good' = repaid = APPROVE, 'bad' = defaulted = REJECT
raw['approved'] = (raw['class'] == 'good').astype(int)
y = raw['approved']
X_raw = raw.drop(columns=['class', 'approved'])
X = pd.get_dummies(X_raw, drop_first=True).astype(float)

print(f'{len(raw)} loan applications, {X.shape[1]} encoded features')
print(f'approval rate: {y.mean():.1%}')
X_raw.head(3)

1000 loan applications, 48 encoded features
approval rate: 70.0%


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes


In [2]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

# The bank's production model: accurate, opaque.
model = RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
model.fit(X_tr, y_tr)
print(f'accuracy: {accuracy_score(y_te, model.predict(X_te)):.3f}')
print(f'roc_auc : {roc_auc_score(y_te, model.predict_proba(X_te)[:,1]):.3f}')
print('\nThis model decides who gets a loan. It cannot explain itself. That is the problem.')

accuracy: 0.776
roc_auc : 0.799

This model decides who gets a loan. It cannot explain itself. That is the problem.


## Audience 1 - The loan officer: a **global**, **intrinsic** explanation

The officer does not want a per-customer heat map. They want the bank's *policy* in their head:
what generally gets you approved. So we do not explain the black box at all - we **replace it**
with a model that is transparent by design (Molnar's intrinsic route). The cost is a little accuracy.
That cost is the module's central trade-off, priced explicitly.

In [3]:
# A) coefficients: readable effect per feature
sc = StandardScaler().fit(X_tr)
lr = LogisticRegression(max_iter=3000, random_state=SEED).fit(sc.transform(X_tr), y_tr)
print(f'LogReg  accuracy {accuracy_score(y_te, lr.predict(sc.transform(X_te))):.3f} '
      f'| roc_auc {roc_auc_score(y_te, lr.predict_proba(sc.transform(X_te))[:,1]):.3f}')

coef = pd.Series(lr.coef_[0], index=X.columns).sort_values()
print('\n--- Pushes you toward REJECTION ---')
print(coef.head(6).round(3).to_string())
print('\n--- Pushes you toward APPROVAL ---')
print(coef.tail(6).round(3).to_string()[::1])

fig, ax = plt.subplots(figsize=(8, 5))
top = pd.concat([coef.head(8), coef.tail(8)])
ax.barh(top.index, top.values,
        color=['#c0392b' if v < 0 else '#27ae60' for v in top.values])
ax.axvline(0, color='k', lw=.8)
ax.set_title('Loan officer view - GLOBAL: what drives approval (logistic coefficients)')
ax.set_xlabel('<- rejection      coefficient      approval ->')
fig.savefig(OUT / 'a2_global_coefficients.png', dpi=130, bbox_inches='tight')
plt.show()

LogReg  accuracy 0.720 | roc_auc 0.767

--- Pushes you toward REJECTION ---
credit_amount            -0.380
housing_rent             -0.346
purpose_new car          -0.325
duration                 -0.316
installment_commitment   -0.278
foreign_worker_yes       -0.274

--- Pushes you toward APPROVAL ---
savings_status_no known savings                  0.229
other_payment_plans_none                         0.277
other_parties_guarantor                          0.359
purpose_used car                                 0.451
credit_history_critical/other existing credit    0.594
checking_status_no checking                      0.610


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18225/3218000279.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# B) the same policy as RULES - what a loan officer can actually carry in their head
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_tr, y_tr)
print(f'Depth-3 tree accuracy: {accuracy_score(y_te, tree.predict(X_te)):.3f}\n')
print(export_text(tree, feature_names=list(X.columns), max_depth=3))
print('That is the whole model. A human can audit it. That is what "intrinsic" buys you.')

Depth-3 tree accuracy: 0.708

|--- checking_status_no checking <= 0.50
|   |--- duration <= 22.50
|   |   |--- credit_amount <= 1285.00
|   |   |   |--- class: 1
|   |   |--- credit_amount >  1285.00
|   |   |   |--- class: 1
|   |--- duration >  22.50
|   |   |--- purpose_used car <= 0.50
|   |   |   |--- class: 0
|   |   |--- purpose_used car >  0.50
|   |   |   |--- class: 1
|--- checking_status_no checking >  0.50
|   |--- other_payment_plans_none <= 0.50
|   |   |--- purpose_new car <= 0.50
|   |   |   |--- class: 1
|   |   |--- purpose_new car >  0.50
|   |   |   |--- class: 0
|   |--- other_payment_plans_none >  0.50
|   |   |--- age <= 23.50
|   |   |   |--- class: 1
|   |   |--- age >  23.50
|   |   |   |--- class: 1

That is the whole model. A human can audit it. That is what "intrinsic" buys you.


## Audience 2 - The rejected customer: **CEM** (contrastive / *pertinent negative*)

This is the explanation that actually matters to a human being, and it is the one banks are worst at.
The customer does not want feature importances. They want:

> **"What is the smallest change that would have got me approved?"**

That is CEM's *pertinent negative*: the minimal perturbation that flips the decision. We search for
it directly over the **actionable** features - because telling someone to change their age is not
an explanation, it is an insult.

In [5]:
# find a rejected applicant
pred_te = model.predict(X_te)
rejected_pos = np.where(pred_te == 0)[0][3]
applicant = X_te.iloc[[rejected_pos]]
p0 = model.predict_proba(applicant)[0, 1]
print(f'Applicant #{X_te.index[rejected_pos]} -> P(approve) = {p0:.3f}  ==> REJECTED\n')

orig = X_raw.loc[X_te.index[rejected_pos]]
print(orig[['duration', 'credit_amount', 'savings_status', 'checking_status',
            'employment', 'age', 'purpose']].to_string())

Applicant #914 -> P(approve) = 0.410  ==> REJECTED

duration                 24
credit_amount          3161
savings_status         <100
checking_status          <0
employment           1<=X<4
age                      31
purpose            business


In [6]:
# CEM-style search: which single ACTIONABLE change flips the decision?
# Actionable = the applicant could plausibly alter it. Age/sex/foreign_worker are NOT.
ACTIONABLE = ['duration', 'credit_amount', 'installment_commitment',
              'existing_credits', 'num_dependents', 'residence_since']

results = []
for feat in ACTIONABLE:
    lo, hi = X[feat].quantile([0.02, 0.98])
    for val in np.linspace(lo, hi, 60):
        cand = applicant.copy()
        cand[feat] = val
        p = model.predict_proba(cand)[0, 1]
        if p >= 0.5:
            results.append({'change': feat,
                            'from': applicant[feat].values[0],
                            'to': round(val, 1),
                            'P(approve)': round(p, 3),
                            'effort (|sd|)': round(abs(val - applicant[feat].values[0])
                                                   / X[feat].std(), 2)})
            break

if results:
    cem = pd.DataFrame(results).sort_values('effort (|sd|)')
    print('PERTINENT NEGATIVES - single changes that flip REJECTED -> APPROVED:\n')
    print(cem.to_string(index=False))
    best = cem.iloc[0]
    print(f"\n>> The explanation you give the customer:\n"
          f"   'Your application was declined. Had your {best['change']} been "
          f"{best['to']} instead of {best['from']}, it would have been approved.'")
else:
    print('No single actionable change flips this decision - the rejection is over-determined.')
    print('That is itself an important (and honest) explanation to give.')

PERTINENT NEGATIVES - single changes that flip REJECTED -> APPROVED:

  change  from  to  P(approve)  effort (|sd|)
duration  24.0 6.0        0.51           1.49

>> The explanation you give the customer:
   'Your application was declined. Had your duration been 6.0 instead of 24.0, it would have been approved.'


## Audience 3 - The data scientist: **SHAP** (local attribution)

The DS does not want a counterfactual, they want the **decomposition**: how much did each feature
contribute to *this* score, and do those contributions sum to the prediction? That is exactly what
Shapley values guarantee (Molnar Ch. 4).

In [7]:
import shap

explainer = shap.TreeExplainer(model)
sv = explainer(X_te)
sv_pos = sv[..., 1] if sv.values.ndim == 3 else sv  # binary RF -> two output dims

one = sv_pos[rejected_pos]
contrib = pd.DataFrame({'feature': X.columns, 'shap': one.values}) \
            .assign(absv=lambda d: d.shap.abs()) \
            .sort_values('absv', ascending=False).head(10)

print(f'base rate (avg applicant): {one.base_values:.3f}')
print(f'this applicant           : {one.base_values + one.values.sum():.3f}\n')
print('Top contributions to THIS rejection (negative = pushed toward reject):')
print(contrib[['feature', 'shap']].round(3).to_string(index=False))

plt.figure()
shap.plots.waterfall(one, max_display=12, show=False)
plt.title('Data scientist view - LOCAL: SHAP for this one rejection', fontsize=10)
plt.savefig(OUT / 'a2_shap_waterfall.png', dpi=130, bbox_inches='tight')
plt.show()

/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


base rate (avg applicant): 0.700
this applicant           : 0.410

Top contributions to THIS rejection (negative = pushed toward reject):
                                      feature   shap
                           checking_status_<0 -0.087
                  checking_status_no checking -0.072
credit_history_critical/other existing credit -0.035
                                 housing_rent -0.030
                          savings_status_<100 -0.029
                                  housing_own -0.027
                              purpose_new car  0.021
                                credit_amount  0.021
                       installment_commitment -0.016
                 credit_history_existing paid -0.015


/var/folders/c4/t_n58l316yl308hxjf5jlwsh0000gn/T/ipykernel_18225/3533196104.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Audience 4 - The regulator: **Protodash** (explanation by precedent)

Protodash answers a question no attribution method can: **"show me the applicants most like this
one, and what happened to them."** It explains by *precedent*, not by arithmetic - which is, not
coincidentally, how law works.

It greedily picks prototypes that best *represent* the target - here, the approved applicants most
similar to our rejected one. If they look nearly identical, the bank has a consistency problem.

In [8]:
def protodash(target, candidates, k=3, gamma=None):
    """Greedy prototype selection with an RBF kernel (the core of Protodash).
    Picks the k candidates that jointly best represent `target`."""
    if gamma is None:
        gamma = 1.0 / candidates.shape[1]
    d = np.linalg.norm(candidates - target, axis=1)
    sim = np.exp(-gamma * d ** 2)          # similarity to the target
    chosen, covered = [], np.zeros(len(candidates))
    for _ in range(k):
        gain = sim - covered               # marginal, so prototypes stay diverse
        gain[chosen] = -np.inf
        j = int(np.argmax(gain))
        chosen.append(j)
        dj = np.linalg.norm(candidates - candidates[j], axis=1)
        covered = np.maximum(covered, np.exp(-gamma * dj ** 2) * sim[j])
    return chosen

sc_all = StandardScaler().fit(X_tr)
approved_mask = (model.predict(X_tr) == 1)
cand = sc_all.transform(X_tr[approved_mask])
target = sc_all.transform(applicant)[0]

proto_idx = protodash(target, cand, k=3)
proto_rows = X_tr[approved_mask].iloc[proto_idx]

show = ['duration', 'credit_amount', 'installment_commitment', 'age', 'existing_credits']
table = pd.concat([applicant[show], proto_rows[show]])
table.index = ['>> REJECTED applicant'] + [f'approved precedent {i+1}' for i in range(3)]
table['P(approve)'] = model.predict_proba(pd.concat([applicant, proto_rows]))[:, 1].round(3)

print('PROTODASH - the approved applicants most similar to our rejected one:\n')
print(table.round(1).to_string())
print('\nIf these look almost identical to the rejected applicant, the bank has a problem:')
print('the decision boundary is doing something it cannot justify to a regulator.')

PROTODASH - the approved applicants most similar to our rejected one:

                       duration  credit_amount  installment_commitment   age  existing_credits  P(approve)
>> REJECTED applicant      24.0         3161.0                     4.0  31.0               1.0         0.4
approved precedent 1       24.0         1382.0                     4.0  26.0               2.0         0.9
approved precedent 2       21.0         2606.0                     4.0  28.0               1.0         0.8
approved precedent 3       15.0         2687.0                     2.0  26.0               1.0         0.9

If these look almost identical to the rejected applicant, the bank has a problem:
the decision boundary is doing something it cannot justify to a regulator.


## Discussion (the brief's question)

**"Assume the bank has a legal obligation to explain every rejection. Can you? What is missing?"**

We can now produce four true statements about the same rejection:

1. *Globally*, short loan duration and a healthy checking balance drive approvals (coefficients/rules).
2. *For you*, the biggest negative contributions were X, Y, Z (SHAP).
3. *To flip it*, reduce the loan duration to N months (CEM pertinent negative).
4. *For consistency*, here are 3 approved applicants who look like you (Protodash).

**Only #3 is an explanation a human can act on.** The others are descriptions of a model, not
reasons for a decision - which is precisely the gap the module is pointing at.

**What is still missing, and it is a lot:**

- **Explaining the model is not explaining the decision.** SHAP faithfully reports what the model
  did. It is silent on whether the model *should* have done it. A model that learnt a proxy for a
  protected attribute produces a beautiful, fully-legible, discriminatory explanation.
- **The proxy problem is live in this dataset.** `personal_status` encodes sex and
  `foreign_worker` is a protected class - both sit in the feature matrix and both are unactionable.
  A counterfactual that says *"be a different person"* is not a remedy.
- **Counterfactuals are not unique.** Several different single changes flip the decision (see the CEM
  table). The bank picks which one to tell you. That choice is unaudited and unregulated.
- **A confident wrong explanation is worse than none** (Aha, Res. 4). Give a customer an actionable
  reason, have them act on it, and get rejected again - you have destroyed the trust you were
  trying to build.

This is exactly why Sowden (Res. 5) answers the trust problem with an **Algorithm Charter** and a
**Data Ethics Advisory Group** rather than a better plotting library. **Explainability is a
governance problem with a technical component - not a technical problem.**